# Part 7: Recommender Systems

**Course:** 2026 KMITL Data Analytics

A recommender helps a user find useful items in a larger catalog. In this lesson we recommend fictional movies using movie features and other users' ratings. We build content-based, user-based and item-based recommendations, then evaluate hidden ratings without allowing them into training. All data is created in code and runs on a CPU.


## Learning objectives

By the end you can:
1. Build a user-item matrix while keeping missing ratings distinct from dislike.
2. Calculate cosine similarity and build a content-based user profile.
3. Recommend unseen movies using similar users or similar items.
4. Distinguish memory-based methods from model-based matrix factorization.
5. Evaluate rating predictions and a top-K list using held-out observations.
6. Explain sparse ratings, both types of cold start, popularity bias and filter bubbles.


## Required imports

We use NumPy and pandas for matrices, scikit-learn for cosine similarity, and Matplotlib and Seaborn for charts. No movie service, account, download or extra recommendation library is needed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries loaded.')


## Dataset introduction

A **user** receives recommendations; an **item** is something we can recommend. Our six fictional users rate eight fictional movies from 1 (strong dislike) to 5 (strong like). These are **explicit preferences**. Clicks and viewing time are implicit signals: a click can show interest but not satisfaction.

A **user-item matrix** has one row per user and one column per movie. A missing entry means we have not observed a rating. It does not mean zero stars, dislike, or that the user has never watched the movie. For this lesson, 'unseen' means unrated in training. We also have four binary content features: science fiction, action, drama and comedy. A movie can have several features. These hand-written features are available independently of ratings.


In [ ]:
movies = ['Space Rescue', 'Moon Mission', 'Robot City', 'Family Table',
          'Garden Days', 'Home Again', 'Orbit School', 'World Journey']
users = ['An', 'Ben', 'Cara', 'Dan', 'Eva', 'Fah']
rating_matrix = pd.DataFrame([
    [5, 4, np.nan, 1, 2, np.nan, 5, np.nan],
    [4, 5, 4, np.nan, 1, 2, np.nan, np.nan],
    [1, np.nan, 2, 5, 4, 5, np.nan, np.nan],
    [np.nan, 2, 1, 4, 5, 4, np.nan, 3],
    [5, 4, 5, np.nan, np.nan, 1, 4, np.nan],
    [2, np.nan, 1, 5, 4, 4, np.nan, 5],
], index=users, columns=movies, dtype=float)
movie_features = pd.DataFrame([
    [1, 1, 0, 0], [1, 1, 0, 0], [1, 0, 1, 0], [0, 0, 1, 1],
    [0, 0, 1, 0], [0, 0, 1, 1], [1, 0, 0, 1], [0, 1, 1, 0],
], index=movies, columns=['science fiction', 'action', 'drama', 'comedy'])
# Convert observed ratings to records, then pivot them back to a user-item matrix.
rating_records = rating_matrix.rename_axis('user').reset_index().melt(
    id_vars='user', var_name='movie', value_name='rating').dropna()
rebuilt_matrix = rating_records.pivot(index='user', columns='movie', values='rating').reindex(index=users, columns=movies)
assert np.allclose(rebuilt_matrix, rating_matrix, equal_nan=True)
assert rating_records['rating'].between(1, 5).all()
print(rating_records.head().to_string(index=False))
print('Movie features:')
print(movie_features)


## 1. Hide observations before building any recommender

We hold out one known rating per user using a fixed list chosen for this teaching example. The remaining ratings form the training matrix. All rating-derived profiles, similarities, popularity counts and fallback means below use **training data only**. The held-out values are read only for evaluation.

This is a tiny demonstration split, not a realistic benchmark. In a real application, prefer a time-based split: past behavior predicts later behavior, and the candidate catalog must contain only items available at that time. Use separate validation data for choosing methods and settings, then test once.

**Sparsity** means many matrix entries are unknown. If 12 of 20 entries are observed, 8/20 = **40%** are missing. With M missing entries and N total entries, missing fraction = M/N. Here we report it after holding out ratings.


In [ ]:
hidden_movies = ['Moon Mission', 'Garden Days', 'Home Again', 'Robot City', 'Orbit School', 'World Journey']
test_ratings = pd.DataFrame([{'user': user, 'movie': movie, 'actual': rating_matrix.loc[user, movie]}
                             for user, movie in zip(users, hidden_movies)])
train_ratings = rating_matrix.copy()
for row in test_ratings.itertuples(index=False):
    train_ratings.loc[row.user, row.movie] = np.nan
assert test_ratings['actual'].notna().all()
assert all(pd.isna(train_ratings.loc[row.user, row.movie]) for row in test_ratings.itertuples(index=False))
assert train_ratings.notna().sum().sum() + len(test_ratings) == len(rating_records)
print(f'Training missing fraction: {train_ratings.isna().to_numpy().mean():.1%}')
fig, ax = plt.subplots(figsize=(11, 4))
ax.set_facecolor('#dddddd')
sns.heatmap(train_ratings, mask=train_ratings.isna(), annot=True, fmt='.0f',
            cmap='YlGnBu', vmin=1, vmax=5, linewidths=0.5, cbar_kws={'label': 'Observed rating'}, ax=ax)
ax.set(title='Training user-item matrix (gray = unknown)', xlabel='Movie', ylabel='User')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


**Read the heatmap:** colored cells are observed training ratings and gray cells are unknown, including the held-out observations. Some users favor science-fiction movies, while others favor family dramas. There are too few observations to infer a stable preference for every person. The matrix is deliberately small enough to inspect by eye.


## 2. Content-based recommendation and cosine similarity

Content-based recommendation compares item features. Consider Space Rescue with vector [1, 1, 0, 0] and Robot City with [1, 0, 1, 0]. Multiply corresponding entries and add: 1 + 0 + 0 + 0 = **1**, the dot product. Each vector has length the square root of 2. Dividing 1 by their length product gives **0.5**. Two identical nonzero feature vectors have cosine similarity 1.

For feature vectors x and y with p entries, j names an entry and ||x|| is its Euclidean length:
$$\operatorname{cosine}(x,y)=\frac{\sum_{j=1}^{p}x_jy_j}{\sqrt{\sum_{j=1}^{p}x_j^2}\sqrt{\sum_{j=1}^{p}y_j^2}}.$$

Cosine measures direction rather than size. With these nonnegative features it ranges from 0 to 1; general signed vectors can give values down to −1. A zero vector has no direction, so the mathematical expression is undefined; scikit-learn returns zero similarity for it. Feature quality matters: identical genre tags do not make two movies identical experiences.


In [ ]:
content_similarity = pd.DataFrame(cosine_similarity(movie_features), index=movies, columns=movies)
assert np.isclose(content_similarity.loc['Space Rescue', 'Robot City'], 0.5)
assert np.allclose(np.diag(content_similarity), 1)
similar_movies = content_similarity.loc['Space Rescue'].drop('Space Rescue').sort_values(ascending=False, kind='stable')
print('Movies similar to Space Rescue (not yet personalized):')
print(similar_movies.head(3))


### A user profile makes content recommendations personal

A **user profile** summarizes the features of movies a user likes. We define liked as a training rating of at least 4, and average those feature vectors. For two liked movies [1, 1, 0, 0] and [1, 0, 0, 1], the mean profile is **[1, 0.5, 0, 0.5]**: strong science-fiction interest, some action and comedy.

Let L_u be the set of liked training movies for user u, |L_u| its size, and f_i movie i's feature vector. The mean profile p_u is:
$$p_u=\frac{1}{|L_u|}\sum_{i\in L_u}f_i.$$

We rank unrated movies by cosine similarity to this profile. The score is neither a probability nor a predicted star rating. This simple profile ignores negative preferences and gives each liked movie equal weight. If there are no liked movies, we return no personalized content result and use a clearly labeled fallback later.


In [ ]:
def content_scores(user, ratings):
    liked = ratings.loc[user].dropna()
    liked = liked[liked >= 4].index
    unseen = ratings.columns[ratings.loc[user].isna()]
    if len(liked) == 0:
        return pd.Series(dtype=float, name='content similarity')
    profile = movie_features.loc[liked].mean(axis=0).to_numpy().reshape(1, -1)
    scores = cosine_similarity(profile, movie_features.loc[unseen]).ravel() if len(unseen) else np.array([])
    return pd.Series(scores, index=unseen, name='content similarity').sort_values(ascending=False, kind='stable')

an_content = content_scores('An', train_ratings)
print('An: content-based recommendations')
print(an_content)
assert set(an_content.index).isdisjoint(train_ratings.loc['An'].dropna().index)
assert an_content.between(0, 1 + 1e-12).all()


## 3. Collaborative filtering: learn from shared behavior

**Collaborative filtering** does not need genre features. It uses the rating patterns of users and items. **User-based filtering** asks which users have similar observed tastes, then uses their ratings. **Item-based filtering** asks which items have similar rating patterns, then uses the target user's ratings of those items.

We use a simple, uncentered cosine baseline. For similarity calculation only, we fill missing entries with zero. Zero is a placeholder for no contribution to the dot product, **not a real rating or a dislike**. Different overlap patterns still affect the cosine norms, so sparse overlap can give weak evidence. We keep the original missing-value mask when predicting: only observed ratings belong in the weighted average.

For example, two neighbors with similarities 0.8 and 0.2 rated a candidate 5 and 3. Its weighted prediction is (0.8 × 5 + 0.2 × 3)/(0.8 + 0.2) = **4.6 stars**. If the second neighbor did not rate the movie, exclude both its rating and its weight, giving 5 stars, not 4.

Let N be neighbors who rated item i, w_uv their similarity to user u, and r_vi their observed rating. The estimated rating is r-hat_ui:
$$\hat r_{ui}=\frac{\sum_{v\in N}w_{uv}r_{vi}}{\sum_{v\in N}w_{uv}}.$$

Exclude the user themself. With no positive-weight evidence, fall back to the item's training mean, then the global training mean if the item has no ratings. This baseline does not correct for generous versus strict raters; centered ratings and overlap-based reliability weights are possible improvements when more data is available.


In [ ]:
def collaborative_scores(ratings, method):
    if method not in ('user', 'item'):
        raise ValueError('method must be user or item')
    observed = ratings.notna().to_numpy()
    values = ratings.fillna(0).to_numpy()
    if not observed.any():
        raise ValueError('At least one observed training rating is needed')
    if not np.isfinite(values[observed]).all() or not ((values[observed] >= 1) & (values[observed] <= 5)).all():
        raise ValueError('Observed ratings must be finite and between 1 and 5')
    # ponytail: full pairwise similarities suit this tiny matrix; use sparse top-K neighbors for a large catalog.
    if method == 'user':
        similarity = cosine_similarity(values)
        np.fill_diagonal(similarity, 0)
        numerator = similarity @ values
        denominator = similarity @ observed.astype(float)
    else:
        similarity = cosine_similarity(values.T)
        np.fill_diagonal(similarity, 0)
        numerator = values @ similarity
        denominator = observed.astype(float) @ similarity
    item_mean = ratings.mean(axis=0).fillna(values[observed].mean()).to_numpy()
    predictions = np.broadcast_to(item_mean, values.shape).copy()
    np.divide(numerator, denominator, out=predictions, where=denominator > 0)
    return pd.DataFrame(predictions, index=ratings.index, columns=ratings.columns)

def recommend(user, predictions, ratings, k=3):
    unseen = ratings.columns[ratings.loc[user].isna()]
    return predictions.loc[user, unseen].sort_values(ascending=False, kind='stable').head(k)

user_similarity = pd.DataFrame(cosine_similarity(train_ratings.fillna(0)), index=users, columns=users)
print('Users most similar to An:')
print(user_similarity.loc['An'].drop('An').sort_values(ascending=False))
user_predictions = collaborative_scores(train_ratings, 'user')
print('An: user-based predictions for unrated movies')
print(recommend('An', user_predictions, train_ratings))
assert np.isclose((0.8 * 5 + 0.2 * 3) / (0.8 + 0.2), 4.6)


### Item-based filtering is not content similarity

Content similarity compared genre columns. Collaborative item similarity compares columns of ratings across users. Two movies can be collaboratively similar even if their genres differ. The same cosine calculation is applied to the transposed training matrix.

If an unrated movie has similarities 0.75 and 0.25 to movies a user rated 4 and 2, its predicted rating is (0.75 × 4 + 0.25 × 2)/(0.75 + 0.25) = **3.5**. In general, O_u contains the items observed for user u, s_ij is the similarity between items i and j, and r_uj is the user's observed rating for item j:
$$\hat r_{ui}=\frac{\sum_{j\in O_u}s_{ij}r_{uj}}{\sum_{j\in O_u}s_{ij}}.$$

The code above implements both weighted averages with matrix multiplication. It excludes an item's self-similarity and uses the same fallback rule. Both methods use all positive-similarity neighbors here; a production system usually retrieves a smaller neighbor set. Stable sorting makes ties follow the catalog order, not evidence of a better preference match.


In [ ]:
item_similarity = pd.DataFrame(cosine_similarity(train_ratings.fillna(0).T), index=movies, columns=movies)
print('Collaboratively similar to Space Rescue:')
print(item_similarity.loc['Space Rescue'].drop('Space Rescue').sort_values(ascending=False).head(3))
item_predictions = collaborative_scores(train_ratings, 'item')
an_user = recommend('An', user_predictions, train_ratings, k=8)
an_item = recommend('An', item_predictions, train_ratings, k=8)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
an_content.plot.bar(ax=axes[0], color='teal', label='Content similarity')
axes[0].set(title='An: content ranking', xlabel='Unrated movie', ylabel='Cosine similarity', ylim=(0, 1.05))
pd.DataFrame({'User-based': an_user, 'Item-based': an_item}).plot.bar(ax=axes[1])
axes[1].set(title='An: collaborative scores', xlabel='Unrated movie', ylabel='Predicted stars', ylim=(0, 5.3))
for ax in axes:
    ax.tick_params(axis='x', rotation=35)
    ax.legend()
plt.tight_layout()
plt.show()


**Read the score charts:** the left panel ranks feature matches to An's liked movies. The right compares star estimates from users and items. Differences come from different evidence: genre features versus observed behavior. Do not compare the heights across panels as though cosine and stars were the same unit. Only movies unrated in training are candidates, including the hidden movie; this is intentional for offline evaluation.


## 4. Memory-based versus model-based methods

The two collaborative methods above are **memory-based**: they keep the rating matrix and compare neighbors. **Model-based** methods learn parameters that summarize behavior. In **matrix factorization**, each user and each movie has a short vector of learned, hidden interests, called latent factors. These factors need not match named genres.

A user vector [1, 0.5] and item vector [4, 2] produce a dot product of 1 × 4 + 0.5 × 2 = **5**. For matrices P (user vectors) and Q (item vectors), the estimated rating matrix is:
$$\hat R=PQ^T.$$

Here R-hat contains estimated ratings, and the transpose Q^T turns movie rows into columns. Real factorization learns vectors by fitting **observed training entries only**, often with user/item biases and regularization. Unknown ratings are not zero-valued training targets. Predictions may fall outside the star range unless the model handles that. The next cell is a forward calculation with chosen vectors, **not a trained recommender**. A full factorization trainer is not needed to understand the concept.


In [ ]:
user_factors = np.array([[1., 0.5], [0.2, 1.]])
movie_factors = np.array([[4., 2.], [1., 3.], [2., 1.]])
factor_predictions = user_factors @ movie_factors.T
print(pd.DataFrame(factor_predictions, index=['Example user A', 'Example user B'],
                   columns=['Example movie 1', 'Example movie 2', 'Example movie 3']))
assert factor_predictions.shape == (2, 3)
assert np.isclose(factor_predictions[0, 0], 5)


## 5. Evaluation: stars and ranked lists answer different questions

**Mean Absolute Error (MAE)** measures rating prediction error. If hidden ratings are 5 and 2 and predictions are 4 and 3, the absolute errors are 1 and 1, so MAE = **1 star**. For T held-out pairs, actual rating r_t and prediction r-hat_t:
$$MAE=\frac{1}{T}\sum_{t=1}^{T}|r_t-\hat r_t|.$$

We compare user-based and item-based predictions with an item-mean baseline computed from training ratings. All six held-out ratings, including low ratings, are used for MAE. Content cosine scores are not star estimates and are not included in MAE.

For ranked lists, define relevant as a held-out rating of at least 4. If a top-3 list contains one of two known relevant items, **precision@3** is 1/3 and **recall@3** is 1/2. Let H be the number of retrieved relevant items, K the list length, and L the number of held-out relevant items:
$$precision@K=H/K,\qquad recall@K=H/L.$$

Here each eligible user has just one held-out relevant movie, so recall@K is a hit indicator. Users whose only held-out rating is below 4 are excluded from recall, not assigned an invented zero denominator. Other unknown candidates may be relevant too, so these metrics measure recovery of **known held-out** preferences, not complete user satisfaction.


In [ ]:
global_mean = train_ratings.stack().mean()
baseline_means = train_ratings.mean(axis=0).fillna(global_mean)
baseline_predictions = pd.DataFrame(np.tile(baseline_means.to_numpy(), (len(users), 1)), index=users, columns=movies)
models = {'Item mean': baseline_predictions, 'User-based': user_predictions, 'Item-based': item_predictions}
mae_rows = []
for name, predictions in models.items():
    hidden_predictions = np.array([predictions.loc[row.user, row.movie] for row in test_ratings.itertuples(index=False)])
    error = np.abs(test_ratings['actual'].to_numpy() - hidden_predictions).mean()
    mae_rows.append({'method': name, 'test MAE': error, 'test ratings': len(test_ratings)})
print(pd.DataFrame(mae_rows).round(3).to_string(index=False))

top_k = 2
relevant_test = test_ratings[test_ratings['actual'] >= 4]
ranking_rows = []
for name in ['Content', 'Item mean', 'User-based', 'Item-based']:
    hits = []
    for row in relevant_test.itertuples(index=False):
        scores = content_scores(row.user, train_ratings).head(top_k) if name == 'Content' else recommend(row.user, models[name], train_ratings, top_k)
        assert len(scores) == top_k
        hits.append(int(row.movie in scores.index))
    ranking_rows.append({'method': name, 'precision@2': np.mean(hits) / top_k,
                         'recall@2': np.mean(hits), 'eligible users': len(hits)})
print('Known-positive retrieval:')
print(pd.DataFrame(ranking_rows).round(3).to_string(index=False))


**Read the evaluation tables:** lower MAE means closer star estimates on these six hidden ratings. Higher precision and recall mean better retrieval of the four hidden liked movies among training-unrated candidates. With only four eligible users, one hit changes mean recall by 0.25. The apparent winner is not reliable evidence for deployment. Do not repeatedly adjust the method against these test ratings; use validation data for that.

A real evaluation should also check catalog coverage, diversity, different user groups and recent versus older items. An online experiment can measure satisfaction or retention, not just clicks. It requires privacy safeguards and a suitable comparison group. Offline scores alone cannot show the causal effect of a recommendation.


## 6. Popularity, cold start and recommendation risks

**Popularity** here counts observed training ratings per movie, not how many people liked it. For example, ratings [5, 1, unknown] give a count of 2 but a mean of 3. A rating count is evidence of exposure, not quality.

- **New-user cold start:** no personal history means no collaborative neighbors or content profile. Ask for a few preferences, offer a diverse editorial list, or label a popularity fallback clearly.
- **New-item cold start:** a new movie has no collaborative rating vector. Content features can still connect it to existing interests; otherwise it needs exploration or editorial exposure.
- **Sparse ratings:** two users may have little overlap. Similarity can be unstable even when it looks large. Gather more evidence and report uncertainty.
- **Popularity bias:** already visible items collect more interactions, which can cause the system to show them even more.
- **Filter bubbles:** repeatedly showing only familiar content can narrow exposure. Diversity, exploration and user controls can help, while still respecting user preferences.


In [ ]:
popularity = train_ratings.notna().sum().sort_values(ascending=False, kind='stable')
fig, ax = plt.subplots(figsize=(10, 4))
popularity.plot.bar(ax=ax, color='steelblue', label='Training rating count')
ax.set(title='Observed popularity is not the same as liking', xlabel='Movie', ylabel='Number of training ratings')
ax.set_yticks(range(int(popularity.max()) + 1))
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout()
plt.show()


**Read the popularity chart:** taller bars identify movies with more training observations, including dislikes. These movies provide more collaborative evidence and can dominate recommendations. The counts come from a tiny hand-written sample, not real market popularity. Held-out ratings do not contribute to these bars.


In [ ]:
cold_ratings = train_ratings.reindex(index=users + ['New user'], columns=movies + ['New movie'])
cold_predictions = collaborative_scores(cold_ratings, 'user')
print('New user: training-popularity fallback (not personalized)')
print(popularity.head(3))
print('New movie collaborative fallback for An:', round(cold_predictions.loc['An', 'New movie'], 2))
assert np.isclose(cold_predictions.loc['An', 'New movie'], global_mean)
new_movie_features = np.array([[1, 0, 0, 1]])
print('New movie content matches despite having no ratings:')
print(pd.Series(cosine_similarity(new_movie_features, movie_features).ravel(), index=movies).sort_values(ascending=False).head(3))


The new user's list is explicitly a popularity fallback, not inferred taste. The unrated new movie gets only a global-mean collaborative estimate, which is weak evidence. Its known science-fiction and comedy features still identify content matches immediately. Keep these fallback types visible so a numerical output is not mistaken for a well-supported personal recommendation.


## Common mistakes

- Replacing missing ratings with actual zero-star targets instead of retaining an observation mask.
- Calculating similarities or popularity on the full matrix before hiding test ratings.
- Recommending items already rated when the task asks for new discoveries.
- Including the target user or item's own similarity as evidence for itself.
- Calling cosine similarity a probability or comparing it directly with star ratings.
- Assuming a large similarity based on little overlap is reliable.
- Claiming high accuracy from a handful of held-out users, or ignoring low ratings when evaluating MAE.
- Optimizing clicks alone while ignoring exposure bias, diversity, privacy and user control.


In [ ]:
# These checks run with Run all; they guard the important prediction rules.
for name, predictions in models.items():
    assert np.isfinite(predictions.to_numpy()).all()
    assert ((predictions.to_numpy() >= 1 - 1e-12) & (predictions.to_numpy() <= 5 + 1e-12)).all()
    for user in users:
        result = recommend(user, predictions, train_ratings)
        assert set(result.index).isdisjoint(train_ratings.loc[user].dropna().index)

# One identical neighbor supplies a rating; an unknown neighbor rating is excluded.
small_ratings = pd.DataFrame([[5., np.nan], [5., 4.]], index=['Target', 'Neighbor'], columns=['Known', 'Candidate'])
assert np.isclose(collaborative_scores(small_ratings, 'user').loc['Target', 'Candidate'], 4)
small_items = pd.DataFrame([[4., np.nan], [5., 5.]], index=['Target', 'Other'], columns=['Known', 'Candidate'])
assert np.isclose(collaborative_scores(small_items, 'item').loc['Target', 'Candidate'], 4)
for method in ['user', 'item']:
    cold_scores = collaborative_scores(cold_ratings, method)
    assert np.isfinite(cold_scores.to_numpy()).all()
    assert np.isclose(cold_scores.loc['New user', 'New movie'], global_mean)
assert content_scores('New user', train_ratings.reindex(index=users + ['New user'])).empty
assert recommend('An', user_predictions, train_ratings.fillna(3)).empty
# Hiding every original test value again reconstructs exactly the training matrix.
reconstructed_train = rating_matrix.copy()
for row in test_ratings.itertuples(index=False):
    reconstructed_train.loc[row.user, row.movie] = np.nan
pd.testing.assert_frame_equal(reconstructed_train, train_ratings)
print('Recommender lesson checks passed.')


## Summary

Content-based methods match item features to items or user profiles. Collaborative methods transfer evidence between similar users or items while preserving the distinction between missing and observed ratings. Matrix factorization replaces stored neighbor comparisons with learned short vectors. Evaluate on hidden observations with training-only statistics, and distinguish rating error from ranked retrieval. Cold start, sparse evidence, popularity bias and filter bubbles remain important even when code produces a confident-looking score.
